# Understanding NDCG (Normalized Discounted Cumulative Gain)

This tutorial walks through how NDCG is computed from scratch, then verifies the result using PyTorch Ignite's `NDCG` metric.

NDCG is a ranking metric commonly used in information retrieval and recommender systems. Unlike metrics that only check if the right item was retrieved, NDCG rewards models that rank more relevant items **higher** in the list.

By the end of this notebook you will:
- Understand what ground truth and predictions look like for a ranking problem
- Compute DCG and IDCG step by step by hand
- Calculate NDCG manually
- Verify every number matches the Ignite `NDCG` implementation

## Setup

In [1]:
# Install dependencies if needed
# !pip install pytorch-ignite torch
import torch
import math

## 1. The Problem Setup

Imagine a search engine returning 5 documents for a query. Each document has a **relevance score** (ground truth) assigned by a human — higher means more relevant:

| Document | Relevance (ground truth) |
|----------|-------------------------|
| Doc A    | 3 (highly relevant)     |
| Doc B    | 2 (relevant)            |
| Doc C    | 3 (highly relevant)     |
| Doc D    | 0 (not relevant)        |
| Doc E    | 1 (slightly relevant)   |

The model predicts a **score** for each document. The model then ranks documents by these scores (highest score = rank 1):

In [2]:
# Ground truth relevance scores (one query, 5 documents)
# Shape: (1, 5) — batch of 1 query
y_true = torch.tensor([[3.0, 2.0, 3.0, 0.0, 1.0]])

# Model prediction scores for each document
# Higher score = model thinks this doc is more relevant
y_pred = torch.tensor([[0.1, 0.4, 0.35, 0.8, 0.1]])

print("Ground truth relevance:", y_true)
print("Model prediction scores:", y_pred)

Ground truth relevance: tensor([[3., 2., 3., 0., 1.]])
Model prediction scores: tensor([[0.1000, 0.4000, 0.3500, 0.8000, 0.1000]])


## 2. Step 1 — The DCG Helper Function

DCG measures the quality of a ranking. It rewards relevant documents but **discounts** them based on their position — finding a relevant document at rank 1 is worth more than finding it at rank 5.

The formula is:

$$DCG@K = \sum_{i=1}^{K} \frac{2^{rel_i} - 1}{\log_2(i + 1)}$$

Where:
- $rel_i$ is the relevance of the document at rank $i$
- The numerator $2^{rel_i} - 1$ is the **gain** (higher relevance = exponentially higher gain)
- The denominator $\log_2(i+1)$ is the **discount** (lower rank position = larger discount)

We define a single `compute_dcg` function that accepts both `y_true` (relevance scores) and `scores` (the signal used to rank documents). This lets us reuse the same function for both DCG and IDCG:
- **DCG**: pass `scores=y_pred` → ranks by model predictions
- **IDCG**: pass `scores=y_true` → ranks by ground truth (ideal order)

In [3]:
def compute_dcg(y_true, scores, k):
    """Compute DCG@K by ranking y_true according to scores.

    Args:
        y_true:  1D tensor of ground-truth relevance values
        scores:  1D tensor used to rank the documents (descending)
                 Pass y_pred to get DCG; pass y_true to get IDCG.
        k:       number of top positions to consider

    Returns:
        DCG@K score (float)
    """
    # Rank documents by scores (descending) and reorder relevance accordingly
    ranked_indices = torch.argsort(scores, descending=True)
    ranked_relevance = y_true[ranked_indices]

    dcg = 0.0
    print(f"{'Rank':<6} {'Relevance':<12} {'Gain (2^rel-1)':<18} {'Discount log2(i+1)':<22} {'Contribution'}")
    print("-" * 72)
    for i in range(k):
        rank = i + 1
        rel = ranked_relevance[i].item()
        gain = (2 ** rel) - 1
        discount = math.log2(rank + 1)
        contribution = gain / discount
        dcg += contribution
        print(f"{rank:<6} {rel:<12.0f} {gain:<18.4f} {discount:<22.4f} {contribution:.4f}")
    print("-" * 72)
    return dcg

## 3. Step 2 — Compute DCG and IDCG

We call `compute_dcg` twice:
- `compute_dcg(y_true, y_pred, k)` → ranks by model predictions → **DCG**
- `compute_dcg(y_true, y_true, k)` → ranks by ground truth → **IDCG**

In [4]:
K = 5  # Evaluate top 5 results

# --- DCG: rank by model predictions ---
print(f"DCG@{K} — model's ranking (scores = y_pred):\n")
dcg = compute_dcg(y_true[0], y_pred[0], K)
print(f"DCG@{K} = {dcg:.4f}")

print()

# --- IDCG: rank by ground truth (ideal order) ---
print(f"IDCG@{K} — ideal ranking (scores = y_true):\n")
idcg = compute_dcg(y_true[0], y_true[0], K)
print(f"IDCG@{K} = {idcg:.4f}")

DCG@5 — model's ranking (scores = y_pred):

Rank   Relevance    Gain (2^rel-1)     Discount log2(i+1)     Contribution
------------------------------------------------------------------------
1      0            0.0000             1.0000                 0.0000
2      2            3.0000             1.5850                 1.8928
3      3            7.0000             2.0000                 3.5000
4      3            7.0000             2.3219                 3.0147
5      1            1.0000             2.5850                 0.3869
------------------------------------------------------------------------
DCG@5 = 8.7944

IDCG@5 — ideal ranking (scores = y_true):

Rank   Relevance    Gain (2^rel-1)     Discount log2(i+1)     Contribution
------------------------------------------------------------------------
1      3            7.0000             1.0000                 7.0000
2      3            7.0000             1.5850                 4.4165
3      2            3.0000             2.0000

## 4. Step 3 — Compute NDCG

NDCG normalizes DCG by IDCG, giving a score between 0 and 1:

$$NDCG@K = \frac{DCG@K}{IDCG@K}$$

A score of 1.0 means the model ranked everything perfectly. A score close to 0 means the ranking was very poor.

In [5]:
ndcg_manual = dcg / idcg

print(f"DCG@{K}  = {dcg:.4f}")
print(f"IDCG@{K} = {idcg:.4f}")
print(f"NDCG@{K} = DCG / IDCG = {dcg:.4f} / {idcg:.4f} = {ndcg_manual:.4f}")
print(f"\nThe model achieved {ndcg_manual*100:.1f}% of the ideal ranking quality.")

DCG@5  = 8.7944
IDCG@5 = 13.3472
NDCG@5 = DCG / IDCG = 8.7944 / 13.3472 = 0.6589

The model achieved 65.9% of the ideal ranking quality.


## 5. Verify with PyTorch Ignite

Now let's confirm our manual calculation matches the Ignite `NDCG` metric exactly.

In [6]:
from ignite.metrics.rec_sys.ndcg import NDCG

# Initialize the NDCG metric with k=5
ndcg_metric = NDCG(output_transform=lambda x: x, top_k=[K])

# Reset and update with our data
ndcg_metric.reset()
ndcg_metric.update((y_pred, y_true))

# Compute the result
ignite_result = ndcg_metric.compute()

print(f"Manual NDCG@{K}:  {ndcg_manual:.4f}")
print(f"Ignite NDCG@{K}:  {ignite_result[0]:.4f}")
print()

# Verify they match
assert abs(ndcg_manual - ignite_result[0]) < 1e-4, "Mismatch!"
print("✓ Manual calculation matches Ignite implementation perfectly!")

Manual NDCG@5:  0.6589
Ignite NDCG@5:  0.6589

✓ Manual calculation matches Ignite implementation perfectly!


## 6. Understanding the Score

Let's build some intuition by looking at two extreme cases.

In [7]:
# Case 1: Perfect ranking (model scores match relevance exactly)
y_pred_perfect = torch.tensor([[0.9, 0.6, 0.8, 0.1, 0.3]])

ndcg_metric.reset()
ndcg_metric.update((y_pred_perfect, y_true))
perfect_score = ndcg_metric.compute()
print(f"Perfect ranking NDCG@{K}: {perfect_score[0]:.4f} (should be 1.0)")

# Case 2: Worst ranking (model ranks least relevant items highest)
y_pred_worst = torch.tensor([[0.1, 0.3, 0.2, 0.9, 0.6]])

ndcg_metric.reset()
ndcg_metric.update((y_pred_worst, y_true))
worst_score = ndcg_metric.compute()
print(f"Worst ranking  NDCG@{K}: {worst_score[0]:.4f} (should be close to 0)")

print(f"\nOur model's ranking NDCG@{K}: {ndcg_manual:.4f} (somewhere in between)")

Perfect ranking NDCG@5: 1.0000 (should be 1.0)
Worst ranking  NDCG@5: 0.5884 (should be close to 0)

Our model's ranking NDCG@5: 0.6589 (somewhere in between)
